## Section 1: Import Required Libraries and Setup

# Sign Language Dataset Quality Comparison
## VSL vs ASL - Comprehensive Video Analysis Pipeline

This notebook analyzes and compares the visual recording quality between Vietnamese Sign Language (VSL) and American Sign Language (ASL) datasets to determine suitability for sign language recognition models.

**Analysis Metrics:**
- Resolution & Consistency
- Blur/Sharpness (Laplacian variance)
- Lighting Conditions (Brightness mean/std)
- Background Complexity (Edge density)
- Motion & Camera Stability (Optical flow)
- Hand Visibility (Occlusion rate)
- Person Size & Distance (Area ratio)
- Camera Angle & Frontality (Pose-based analysis)

In [15]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
import warnings
import tqdm
from scipy import stats
import json
from pathlib import Path
from tqdm.notebook import tqdm

# Import MediaPipe
import mediapipe as mp

warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"MediaPipe version: {mp.__version__}")

✓ All libraries imported successfully!
OpenCV version: 4.8.1
NumPy version: 1.26.4
Pandas version: 2.2.3
MediaPipe version: 0.10.21


In [ ]:

# ============================================================================
# CONFIGURATION & PATHS
# ============================================================================

# Define dataset paths
PROJECT_ROOT = Path("e:/NAM3/BDML/Project/DBML")
VSL_PATH = PROJECT_ROOT / "data" / "raw" / "VSL"
ASL_PATH = PROJECT_ROOT / "data" / "raw" / "ASL"
OUTPUT_DIR = PROJECT_ROOT / "compare" / "results"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

# Create output directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis parameters tuned for speed.
# Processing all frames in 16k+ videos is extremely slow; these samples are enough
# for dataset-level quality comparison and can be increased for a final run.
SAMPLE_FRAMES = 24          # Number of representative frames per video. Use None only for exhaustive runs.
ANALYSIS_WIDTH = 320        # Resize sampled frames before expensive CV/MediaPipe work. Use None for original size.
FLOW_EVERY_N = 2            # Compute optical flow every N sampled frames.
MEDIAPIPE_EVERY_N = 4       # Run MediaPipe on every N sampled frames.
CHECKPOINT_EVERY = 100      # Save partial CSV every N processed videos.
RESUME_FROM_CHECKPOINT = True
MAX_VIDEOS_PER_DATASET = None  # Set small number such as 100 for smoke tests.

CONFIDENCE_THRESHOLD = 0.5  # MediaPipe confidence threshold
EDGE_THRESHOLD_1 = 50
EDGE_THRESHOLD_2 = 150
BRIGHTNESS_THRESHOLD = 50  # for brightness std
BLUR_THRESHOLD = 100  # for Laplacian variance

# Let OpenCV use optimized kernels. Keeping thread count explicit avoids surprises on some Windows installs.
cv2.setUseOptimized(True)
cv2.setNumThreads(max(1, min(4, os.cpu_count() or 1)))

print(f"Project Root: {PROJECT_ROOT}")
print(f"VSL Dataset: {VSL_PATH}")
print(f"ASL Dataset: {ASL_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"Sample Frames: {SAMPLE_FRAMES or 'All'}")
print(f"Analysis Width: {ANALYSIS_WIDTH or 'Original'}")
print(f"Optical Flow Every N Sampled Frames: {FLOW_EVERY_N}")
print(f"MediaPipe Every N Sampled Frames: {MEDIAPIPE_EVERY_N}")


## Section 2: Dataset Structure Exploration

In [17]:
def scan_dataset_structure(dataset_path: Path, dataset_name: str) -> Dict:
    """
    Recursively scan dataset structure and collect statistics.
    Handles both nested and flat folder structures.
    """
    stats_dict = {
        'dataset_name': dataset_name,
        'root_path': str(dataset_path),
        'total_videos': 0,
        'video_formats': defaultdict(int),
        'folder_structure': [],
        'videos_by_folder': defaultdict(list)
    }
    
    if not dataset_path.exists():
        print(f"❌ Path not found: {dataset_path}")
        return stats_dict
    
    # Recursively find all video files
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv', '.webm'}
    
    for root, dirs, files in os.walk(str(dataset_path)):
        for file in files:
            if Path(file).suffix.lower() in video_extensions:
                stats_dict['total_videos'] += 1
                ext = Path(file).suffix.lower()
                stats_dict['video_formats'][ext] += 1
                
                # Track folder structure
                rel_path = os.path.relpath(root, str(dataset_path))
                stats_dict['videos_by_folder'][rel_path].append(file)
    
    return stats_dict


# Scan both datasets
print("🔍 Scanning dataset structures...")
vsl_stats = scan_dataset_structure(VSL_PATH, "VSL")
asl_stats = scan_dataset_structure(ASL_PATH, "ASL")

print("\n" + "="*80)
print("DATASET EXPLORATION RESULTS")
print("="*80)

for stats in [vsl_stats, asl_stats]:
    print(f"\n📊 {stats['dataset_name']} Dataset Statistics:")
    print(f"   Total Videos: {stats['total_videos']:,}")
    print(f"   Root Path: {stats['root_path']}")
    print(f"   Video Formats:")
    for fmt, count in sorted(stats['video_formats'].items(), key=lambda x: x[1], reverse=True):
        print(f"      {fmt}: {count:,} files")
    print(f"   Folder Structure: {len(stats['videos_by_folder'])} unique folders")
    
print("\n✓ Dataset exploration completed!")

🔍 Scanning dataset structures...

DATASET EXPLORATION RESULTS

📊 VSL Dataset Statistics:
   Total Videos: 4,362
   Root Path: e:\NAM3\BDML\Project\DBML\data\raw\VSL
   Video Formats:
      .mp4: 4,362 files
   Folder Structure: 1 unique folders

📊 ASL Dataset Statistics:
   Total Videos: 11,980
   Root Path: e:\NAM3\BDML\Project\DBML\data\raw\ASL
   Video Formats:
      .mp4: 11,980 files
   Folder Structure: 1 unique folders

✓ Dataset exploration completed!


## Section 3: Video Processing and Frame-Level Metrics Computation

In [ ]:

def resize_for_analysis(frame: np.ndarray) -> np.ndarray:
    """Resize frames before expensive analysis while preserving aspect ratio."""
    if ANALYSIS_WIDTH is None:
        return frame

    height, width = frame.shape[:2]
    if width <= ANALYSIS_WIDTH:
        return frame

    scale = ANALYSIS_WIDTH / width
    new_size = (ANALYSIS_WIDTH, max(1, int(height * scale)))
    return cv2.resize(frame, new_size, interpolation=cv2.INTER_AREA)


def compute_resolution(frame: np.ndarray) -> Tuple[int, int]:
    """Extract frame resolution."""
    height, width = frame.shape[:2]
    return width, height


def compute_basic_frame_metrics(frame: np.ndarray) -> Tuple[float, float, float, float]:
    """Compute brightness, blur, and edge density with a single grayscale conversion."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness_mean = float(np.mean(gray))
    brightness_std = float(np.std(gray))
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, EDGE_THRESHOLD_1, EDGE_THRESHOLD_2)
    edge_density = float(np.count_nonzero(edges) / edges.size)
    return brightness_mean, brightness_std, blur_score, edge_density


def compute_brightness(frame: np.ndarray) -> Tuple[float, float]:
    """Compute brightness statistics (mean and std)."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(np.mean(gray)), float(np.std(gray))


def compute_blur(frame: np.ndarray) -> float:
    """Compute blur/sharpness using Laplacian variance."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compute_edge_density(frame: np.ndarray) -> float:
    """Compute background complexity via Canny edge detection."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, EDGE_THRESHOLD_1, EDGE_THRESHOLD_2)
    return float(np.count_nonzero(edges) / edges.size)


def compute_optical_flow(prev_frame: np.ndarray, curr_frame: np.ndarray) -> float:
    """Compute a lightweight Farneback optical-flow magnitude."""
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        pyr_scale=0.5, levels=1, winsize=9,
        iterations=1, poly_n=5, poly_sigma=1.1, flags=0
    )

    magnitude = cv2.magnitude(flow[..., 0], flow[..., 1])
    return float(np.mean(magnitude))


print("Frame-level metric computation functions defined successfully!")


## Section 4: Hand and Pose Detection with MediaPipe

In [ ]:

# Initialize MediaPipe solutions
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

# Lightweight detector settings for dataset-scale analysis.
hands_detector = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    model_complexity=0,
    min_detection_confidence=CONFIDENCE_THRESHOLD,
    min_tracking_confidence=CONFIDENCE_THRESHOLD
)

pose_detector = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=False,
    min_detection_confidence=CONFIDENCE_THRESHOLD,
    min_tracking_confidence=CONFIDENCE_THRESHOLD
)

print("MediaPipe detectors initialized successfully!")


def compute_hand_metrics(frame: np.ndarray) -> Tuple[bool, float, float]:
    """Compute hand visibility metrics using MediaPipe Hands."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands_detector.process(frame_rgb)

    hand_detected = results.multi_hand_landmarks is not None and len(results.multi_hand_landmarks) > 0
    hand_visibility_ratio = 0.0
    combined_hand_area = 0.0

    if hand_detected:
        total_keypoints = 0
        visible_keypoints = 0

        for hand_landmarks in results.multi_hand_landmarks:
            for landmark in hand_landmarks.landmark:
                total_keypoints += 1
                if landmark.z != 0 or (hasattr(landmark, 'presence') and landmark.presence > CONFIDENCE_THRESHOLD):
                    visible_keypoints += 1

        if total_keypoints > 0:
            hand_visibility_ratio = float(visible_keypoints / total_keypoints)

        x_coords = [lm.x for hand in results.multi_hand_landmarks for lm in hand.landmark]
        y_coords = [lm.y for hand in results.multi_hand_landmarks for lm in hand.landmark]

        if x_coords and y_coords:
            h, w = frame.shape[:2]
            hand_box_area = (max(x_coords) - min(x_coords)) * (max(y_coords) - min(y_coords)) * h * w
            combined_hand_area = float(hand_box_area / (h * w))

    return hand_detected, hand_visibility_ratio, combined_hand_area


def compute_pose_metrics(frame: np.ndarray) -> Tuple[bool, float, float, float, float]:
    """Compute pose-based metrics using MediaPipe Pose."""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose_detector.process(frame_rgb)

    pose_detected = results.pose_landmarks is not None
    person_area_ratio = 0.0
    frontality_score = 0.5
    angle_variance = 0.0
    shoulder_angle = 0.0

    if pose_detected:
        landmarks = results.pose_landmarks.landmark
        x_coords = [lm.x for lm in landmarks if lm.x > 0]
        y_coords = [lm.y for lm in landmarks if lm.y > 0]

        if x_coords and y_coords:
            bbox_width = max(x_coords) - min(x_coords)
            bbox_height = max(y_coords) - min(y_coords)
            person_area_ratio = float(bbox_width * bbox_height)

            left_shoulder = landmarks[11]
            right_shoulder = landmarks[12]
            left_hip = landmarks[23]
            right_hip = landmarks[24]

            if left_shoulder.visibility > CONFIDENCE_THRESHOLD and right_shoulder.visibility > CONFIDENCE_THRESHOLD:
                shoulder_dx = right_shoulder.x - left_shoulder.x
                shoulder_dy = right_shoulder.y - left_shoulder.y
                shoulder_angle = float(np.degrees(np.arctan2(shoulder_dy, shoulder_dx)))

            if (left_shoulder.visibility > CONFIDENCE_THRESHOLD and
                right_shoulder.visibility > CONFIDENCE_THRESHOLD and
                left_hip.visibility > CONFIDENCE_THRESHOLD and
                right_hip.visibility > CONFIDENCE_THRESHOLD):
                center_x = 0.5
                left_s_dist = abs(left_shoulder.x - center_x)
                right_s_dist = abs(right_shoulder.x - center_x)
                left_h_dist = abs(left_hip.x - center_x)
                right_h_dist = abs(right_hip.x - center_x)

                symmetry_score = 1.0 - (abs(left_s_dist - right_s_dist) + abs(left_h_dist - right_h_dist)) / 2
                frontality_score = float(np.clip(symmetry_score, 0, 1))
                angle_variance = float(abs(left_shoulder.y - right_shoulder.y) + abs(left_hip.y - right_hip.y))

    return pose_detected, person_area_ratio, frontality_score, angle_variance, shoulder_angle


In [ ]:

def get_sample_indices(frame_count: int) -> np.ndarray:
    """Return representative frame indices without decoding every frame."""
    if frame_count <= 0:
        return np.array([], dtype=int)
    if SAMPLE_FRAMES is None or SAMPLE_FRAMES >= frame_count:
        return np.arange(frame_count, dtype=int)
    return np.unique(np.linspace(0, frame_count - 1, SAMPLE_FRAMES, dtype=int))


def process_video(video_path: Path) -> Optional[Dict]:
    """Process one video using sampled, resized frames for speed."""
    cap = cv2.VideoCapture(str(video_path))

    try:
        if not cap.isOpened():
            return None

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count == 0:
            return None

        sample_indices = get_sample_indices(frame_count)
        if len(sample_indices) == 0:
            return None

        original_resolution = None
        analysis_resolution = None
        brightness_means = []
        brightness_stds = []
        blur_scores = []
        edge_densities = []
        motions = []
        hand_detected_frames = 0
        hand_visibilities = []
        mediapipe_checked_frames = 0
        person_areas = []
        frontality_scores = []
        angle_variances = []
        shoulder_angles = []
        prev_analysis_frame = None

        for sample_pos, frame_number in enumerate(sample_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
            ret, frame = cap.read()
            if not ret:
                continue

            if original_resolution is None:
                original_resolution = compute_resolution(frame)

            analysis_frame = resize_for_analysis(frame)
            if analysis_resolution is None:
                analysis_resolution = compute_resolution(analysis_frame)

            b_mean, b_std, blur, edge_dens = compute_basic_frame_metrics(analysis_frame)
            brightness_means.append(b_mean)
            brightness_stds.append(b_std)
            blur_scores.append(blur)
            edge_densities.append(edge_dens)

            if prev_analysis_frame is not None and sample_pos % FLOW_EVERY_N == 0:
                motions.append(compute_optical_flow(prev_analysis_frame, analysis_frame))

            if sample_pos % MEDIAPIPE_EVERY_N == 0:
                mediapipe_checked_frames += 1

                hand_det, hand_vis, _ = compute_hand_metrics(analysis_frame)
                if hand_det:
                    hand_detected_frames += 1
                    hand_visibilities.append(hand_vis)

                pose_det, person_area, frontality, angle_var, shoulder_ang = compute_pose_metrics(analysis_frame)
                if pose_det:
                    person_areas.append(person_area)
                    frontality_scores.append(frontality)
                    angle_variances.append(angle_var)
                    shoulder_angles.append(shoulder_ang)

            prev_analysis_frame = analysis_frame

        processed_frames = len(brightness_means)
        if processed_frames == 0:
            return None

        metrics = {
            'video_path': str(video_path),
            'video_name': video_path.name,
            'fps': fps,
            'frame_count': frame_count,
            'processed_frames': processed_frames,
            'sampled_frames': len(sample_indices),
            'sample_frames_config': SAMPLE_FRAMES if SAMPLE_FRAMES is not None else 'all',
            'analysis_width_config': ANALYSIS_WIDTH if ANALYSIS_WIDTH is not None else 'original',
            'flow_every_n': FLOW_EVERY_N,
            'mediapipe_every_n': MEDIAPIPE_EVERY_N,
            'mediapipe_checked_frames': mediapipe_checked_frames,
            'resolution': original_resolution or (0, 0),
            'analysis_resolution': analysis_resolution or (0, 0),
            'brightness_mean': float(np.mean(brightness_means)) if brightness_means else 0.0,
            'brightness_std': float(np.mean(brightness_stds)) if brightness_stds else 0.0,
            'blur_mean': float(np.mean(blur_scores)) if blur_scores else 0.0,
            'blur_std': float(np.std(blur_scores)) if blur_scores else 0.0,
            'edge_density_mean': float(np.mean(edge_densities)) if edge_densities else 0.0,
            'edge_density_std': float(np.std(edge_densities)) if edge_densities else 0.0,
            'motion_mean': float(np.mean(motions)) if motions else 0.0,
            'motion_std': float(np.std(motions)) if motions else 0.0,
            'occlusion_rate': float(1.0 - (hand_detected_frames / mediapipe_checked_frames)) if mediapipe_checked_frames > 0 else 0.0,
            'hand_visibility_ratio': float(np.mean(hand_visibilities)) if hand_visibilities else 0.0,
            'hand_detected_frames': hand_detected_frames,
            'person_area_ratio_mean': float(np.mean(person_areas)) if person_areas else 0.0,
            'person_area_ratio_std': float(np.std(person_areas)) if person_areas else 0.0,
            'frontality_score': float(np.mean(frontality_scores)) if frontality_scores else 0.0,
            'angle_variance': float(np.mean(angle_variances)) if angle_variances else 0.0,
            'shoulder_angle_mean': float(np.mean(shoulder_angles)) if shoulder_angles else 0.0,
        }

        return metrics

    except Exception as e:
        print(f"Error processing {video_path}: {e}")
        return None
    finally:
        cap.release()


print("Video processing function defined successfully!")


## Section 5: Main Pipeline Execution - Process All Videos

In [ ]:

def process_dataset(dataset_path: Path, dataset_name: str, max_videos: Optional[int] = None) -> List[Dict]:
    """Process videos with checkpointing so interrupted runs can resume."""
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv', '.webm'}
    video_files = []

    for root, dirs, files in os.walk(str(dataset_path)):
        for file in files:
            if Path(file).suffix.lower() in video_extensions:
                video_files.append(Path(root) / file)

    video_files = sorted(video_files)
    if max_videos:
        video_files = video_files[:max_videos]

    config_tag = f"sf{SAMPLE_FRAMES or 'all'}_w{ANALYSIS_WIDTH or 'orig'}_mp{MEDIAPIPE_EVERY_N}_flow{FLOW_EVERY_N}"
    checkpoint_path = CHECKPOINT_DIR / f"{dataset_name.lower()}_video_metrics_{config_tag}_checkpoint.csv"
    results = []
    processed_paths = set()

    if RESUME_FROM_CHECKPOINT and checkpoint_path.exists():
        checkpoint_df = pd.read_csv(checkpoint_path)
        if 'video_path' in checkpoint_df.columns:
            checkpoint_df['video_path'] = checkpoint_df['video_path'].astype(str)
            results = checkpoint_df.to_dict('records')
            processed_paths = set(checkpoint_df['video_path'])
            print(f"Resuming {dataset_name}: loaded {len(results):,} checkpointed videos")

    failed_count = 0
    pending_files = [p for p in video_files if str(p) not in processed_paths]

    for idx, video_path in enumerate(tqdm(
        pending_files,
        total=len(pending_files),
        desc=f"Processing {dataset_name}",
        unit="video"
    ), start=1):
        metrics = process_video(video_path)

        if metrics is not None:
            results.append(metrics)
        else:
            failed_count += 1

        if CHECKPOINT_EVERY and idx % CHECKPOINT_EVERY == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)

    if results:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)

    print(f"\n{dataset_name} Processing Complete!")
    print(f"   Total videos discovered: {len(video_files):,}")
    print(f"   Already checkpointed: {len(processed_paths):,}")
    print(f"   Newly attempted: {len(pending_files):,}")
    print(f"   Successfully available: {len(results):,}")
    print(f"   Failed this run: {failed_count}")
    print(f"   Checkpoint: {checkpoint_path}")

    return results


# ===== PROCESS BOTH DATASETS =====
print("Starting dataset processing with sampling + checkpoint resume")
print("="*80)

vsl_results = process_dataset(VSL_PATH / "Videos", "VSL", max_videos=MAX_VIDEOS_PER_DATASET)
asl_results = process_dataset(ASL_PATH / "videos", "ASL", max_videos=MAX_VIDEOS_PER_DATASET)

print("\n" + "="*80)
print("All processing complete!")
print(f"  VSL: {len(vsl_results)} videos available")
print(f"  ASL: {len(asl_results)} videos available")


## Section 6: Video-Level Aggregation

In [ ]:

# Convert results to DataFrames for easier manipulation
vsl_df = pd.DataFrame(vsl_results)
asl_df = pd.DataFrame(asl_results)

print(f"\nVSL DataFrame shape: {vsl_df.shape}")
print(f"ASL DataFrame shape: {asl_df.shape}")

sample_cols = [
    'video_name', 'fps', 'frame_count', 'processed_frames', 'mediapipe_checked_frames',
    'resolution', 'analysis_resolution', 'blur_mean', 'brightness_mean', 'edge_density_mean'
]

def show_sample(df: pd.DataFrame, dataset_name: str) -> None:
    print("\n" + "="*80)
    print(f"SAMPLE VIDEO METRICS ({dataset_name} - First 3 videos)")
    print("="*80)
    if df.empty:
        print("No rows available. Run the processing cell first.")
        return
    available_cols = [col for col in sample_cols if col in df.columns]
    print(df[available_cols].head(3).to_string())

show_sample(vsl_df, 'VSL')
show_sample(asl_df, 'ASL')


## Section 7: Dataset-Level Comparison and Statistics

In [ ]:
def compute_dataset_statistics(df: pd.DataFrame, dataset_name: str) -> Dict:
    """
    Compute aggregated statistics at dataset level.
    """
    stats = {
        'Dataset': dataset_name,
        'Total Videos': len(df),
        
        # Resolution
        'Avg Resolution': str(df['resolution'].iloc[0]) if len(df) > 0 else "N/A",
        
        # Clarity (via blur)
        'Blur Mean (μ)': f"{df['blur_mean'].mean():.2f}",
        'Blur Mean (σ)': f"{df['blur_mean'].std():.2f}",
        
        # Lighting Stability
        'Brightness Std (μ)': f"{df['brightness_std'].mean():.2f}",
        'Brightness Std (σ)': f"{df['brightness_std'].std():.2f}",
        
        # Background Complexity
        'Edge Density (μ)': f"{df['edge_density_mean'].mean():.4f}",
        'Edge Density (σ)': f"{df['edge_density_mean'].std():.4f}",
        
        # Motion/Stability
        'Motion Mean (μ)': f"{df['motion_mean'].mean():.4f}",
        'Motion Mean (σ)': f"{df['motion_mean'].std():.4f}",
        
        # Hand Visibility
        'Occlusion Rate (μ)': f"{df['occlusion_rate'].mean():.4f}",
        'Hand Visibility (μ)': f"{df['hand_visibility_ratio'].mean():.4f}",
        
        # Person Size
        'Person Area (μ)': f"{df['person_area_ratio_mean'].mean():.6f}",
        'Person Area (σ)': f"{df['person_area_ratio_mean'].std():.6f}",
        
        # Camera Angle
        'Frontality Score (μ)': f"{df['frontality_score'].mean():.4f}",
        'Frontality Score (σ)': f"{df['frontality_score'].std():.4f}",
    }
    
    return stats


# Compute statistics for both datasets
vsl_stats = compute_dataset_statistics(vsl_df, 'VSL')
asl_stats = compute_dataset_statistics(asl_df, 'ASL')

print("\n" + "="*100)
print("DATASET-LEVEL STATISTICS")
print("="*100)

# Create comparison DataFrame
stats_df = pd.DataFrame([vsl_stats, asl_stats]).set_index('Dataset').T
print(stats_df.to_string())

print("\n" + "="*100)
print("DETAILED METRIC COMPARISON")
print("="*100)

# Create detailed comparison with numerical values for sorting
comparison_data = {
    'Metric': [
        'Total Videos',
        'Blur Mean',
        'Brightness Stability',
        'Edge Density',
        'Motion Magnitude',
        'Occlusion Rate',
        'Hand Visibility Ratio',
        'Person Area Ratio',
        'Frontality Score'
    ],
    'VSL': [
        len(vsl_df),
        vsl_df['blur_mean'].mean(),
        vsl_df['brightness_std'].mean(),
        vsl_df['edge_density_mean'].mean(),
        vsl_df['motion_mean'].mean(),
        vsl_df['occlusion_rate'].mean(),
        vsl_df['hand_visibility_ratio'].mean(),
        vsl_df['person_area_ratio_mean'].mean(),
        vsl_df['frontality_score'].mean()
    ],
    'ASL': [
        len(asl_df),
        asl_df['blur_mean'].mean(),
        asl_df['brightness_std'].mean(),
        asl_df['edge_density_mean'].mean(),
        asl_df['motion_mean'].mean(),
        asl_df['occlusion_rate'].mean(),
        asl_df['hand_visibility_ratio'].mean(),
        asl_df['person_area_ratio_mean'].mean(),
        asl_df['frontality_score'].mean()
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Add interpretation column
interpretations = [
    '# of videos',
    'Higher is sharper ↑',
    'Lower is more stable ↓',
    'Lower is simpler background ↓',
    'Lower is more stable ↓',
    'Lower is better (less occlusion) ↓',
    'Higher is better ↑',
    'Higher is better (larger person) ↑',
    'Higher is more frontal (better) ↑'
]

comparison_df['Interpretation'] = interpretations

print(comparison_df.to_string(index=False))

# Store for export
final_comparison = comparison_df.copy()
print("\n✓ Dataset comparison computed successfully!")

## Section 8: Results Export and Visualization

In [ ]:
# ===== EXPORT RESULTS =====

# 1. Save dataset-level comparison to CSV
csv_path = OUTPUT_DIR / "dataset_comparison.csv"
final_comparison.to_csv(csv_path, index=False)
print(f"✓ Dataset comparison saved to: {csv_path}")

# 2. Save detailed video-level metrics for VSL
vsl_csv_path = OUTPUT_DIR / "vsl_video_metrics.csv"
vsl_export = vsl_df[['video_name', 'fps', 'frame_count', 'blur_mean', 'brightness_std',
                      'edge_density_mean', 'motion_mean', 'occlusion_rate', 
                      'hand_visibility_ratio', 'person_area_ratio_mean', 
                      'frontality_score']].copy()
vsl_export.to_csv(vsl_csv_path, index=False)
print(f"✓ VSL video metrics saved to: {vsl_csv_path}")

# 3. Save detailed video-level metrics for ASL
asl_csv_path = OUTPUT_DIR / "asl_video_metrics.csv"
asl_export = asl_df[['video_name', 'fps', 'frame_count', 'blur_mean', 'brightness_std',
                      'edge_density_mean', 'motion_mean', 'occlusion_rate', 
                      'hand_visibility_ratio', 'person_area_ratio_mean', 
                      'frontality_score']].copy()
asl_export.to_csv(asl_csv_path, index=False)
print(f"✓ ASL video metrics saved to: {asl_csv_path}")

# 4. Save statistical summary
summary_path = OUTPUT_DIR / "quality_summary.json"
summary_data = {
    'vsl': {
        'total_videos': int(len(vsl_df)),
        'blur_mean': float(vsl_df['blur_mean'].mean()),
        'brightness_std': float(vsl_df['brightness_std'].mean()),
        'edge_density': float(vsl_df['edge_density_mean'].mean()),
        'motion': float(vsl_df['motion_mean'].mean()),
        'occlusion_rate': float(vsl_df['occlusion_rate'].mean()),
        'hand_visibility': float(vsl_df['hand_visibility_ratio'].mean()),
        'person_area': float(vsl_df['person_area_ratio_mean'].mean()),
        'frontality_score': float(vsl_df['frontality_score'].mean()),
    },
    'asl': {
        'total_videos': int(len(asl_df)),
        'blur_mean': float(asl_df['blur_mean'].mean()),
        'brightness_std': float(asl_df['brightness_std'].mean()),
        'edge_density': float(asl_df['edge_density_mean'].mean()),
        'motion': float(asl_df['motion_mean'].mean()),
        'occlusion_rate': float(asl_df['occlusion_rate'].mean()),
        'hand_visibility': float(asl_df['hand_visibility_ratio'].mean()),
        'person_area': float(asl_df['person_area_ratio_mean'].mean()),
        'frontality_score': float(asl_df['frontality_score'].mean()),
    }
}

with open(summary_path, 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f"✓ Summary statistics saved to: {summary_path}")

print(f"\n✓ All results exported to: {OUTPUT_DIR}")

In [ ]:
# ===== VISUALIZATIONS =====

# 1. Bar chart comparison of key metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('VSL vs ASL: Visual Quality Metrics Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = [
    ('blur_mean', 'Blur/Sharpness (↑ Higher = Sharper)', axes[0, 0]),
    ('brightness_std', 'Brightness Stability (↓ Lower = More Stable)', axes[0, 1]),
    ('edge_density_mean', 'Edge Density (↓ Lower = Simpler Background)', axes[0, 2]),
    ('motion_mean', 'Motion Magnitude (↓ Lower = More Stable)', axes[1, 0]),
    ('occlusion_rate', 'Occlusion Rate (↓ Lower = Better Hand Visibility)', axes[1, 1]),
    ('hand_visibility_ratio', 'Hand Visibility Ratio (↑ Higher = Better)', axes[1, 2]),
]

for metric, title, ax in metrics_to_plot:
    vsl_val = vsl_df[metric].mean()
    asl_val = asl_df[metric].mean()
    
    bars = ax.bar(['VSL', 'ASL'], [vsl_val, asl_val], color=['#FF6B6B', '#4ECDC4'], alpha=0.8, edgecolor='black', linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Value')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "quality_comparison_bars.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Bar chart saved to: {OUTPUT_DIR / 'quality_comparison_bars.png'}")

# 2. Additional metrics comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Additional Quality Metrics', fontsize=14, fontweight='bold')

# Person area ratio
ax = axes[0]
vsl_area = vsl_df['person_area_ratio_mean'].mean()
asl_area = asl_df['person_area_ratio_mean'].mean()
bars = ax.bar(['VSL', 'ASL'], [vsl_area, asl_area], color=['#FF6B6B', '#4ECDC4'], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_title('Person Area Ratio (↑ Higher = Better Distance)', fontweight='bold')
ax.set_ylabel('Area Ratio')
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.6f}', ha='center', va='bottom', fontweight='bold')

# Frontality score
ax = axes[1]
vsl_front = vsl_df['frontality_score'].mean()
asl_front = asl_df['frontality_score'].mean()
bars = ax.bar(['VSL', 'ASL'], [vsl_front, asl_front], color=['#FF6B6B', '#4ECDC4'], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_title('Frontality Score (↑ Higher = More Frontal View)', fontweight='bold')
ax.set_ylabel('Frontality Score')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "quality_comparison_additional.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Additional metrics chart saved to: {OUTPUT_DIR / 'quality_comparison_additional.png'}")

In [ ]:
# 3. Radar chart for comprehensive quality assessment
from math import pi

# Normalize metrics to 0-1 scale for radar chart
def normalize_metric(value, metric_name, vsl_mean, vsl_std, asl_mean, asl_std):
    """Normalize metric to 0-1 scale, where 1 is 'better'"""
    
    # Define which direction is "better" for each metric
    higher_is_better = ['blur_mean', 'hand_visibility_ratio', 'frontality_score', 'person_area_ratio_mean']
    lower_is_better = ['brightness_std', 'edge_density_mean', 'motion_mean', 'occlusion_rate']
    
    # Combine ranges from both datasets
    all_values = [vsl_mean - vsl_std, vsl_mean + vsl_std, asl_mean - asl_std, asl_mean + asl_std]
    all_values = [v for v in all_values if v > 0]  # Filter out negative values
    
    if not all_values:
        return 0.5
    
    min_val = min(all_values)
    max_val = max(all_values)
    
    if max_val - min_val < 0.0001:  # Prevent division by zero
        return 0.5
    
    normalized = (value - min_val) / (max_val - min_val)
    
    # Flip if lower is better
    if metric_name in lower_is_better:
        normalized = 1 - normalized
    
    return float(np.clip(normalized, 0, 1))


# Prepare radar chart data
categories = ['Sharpness', 'Lighting\nStability', 'Background\nSimplicity', 
              'Camera\nStability', 'Hand\nVisibility', 'Frontality', 'Person\nSize']

vsl_metrics = [
    normalize_metric(vsl_df['blur_mean'].mean(), 'blur_mean', 
                    vsl_df['blur_mean'].mean(), vsl_df['blur_mean'].std(),
                    asl_df['blur_mean'].mean(), asl_df['blur_mean'].std()),
    normalize_metric(vsl_df['brightness_std'].mean(), 'brightness_std',
                    vsl_df['brightness_std'].mean(), vsl_df['brightness_std'].std(),
                    asl_df['brightness_std'].mean(), asl_df['brightness_std'].std()),
    normalize_metric(vsl_df['edge_density_mean'].mean(), 'edge_density_mean',
                    vsl_df['edge_density_mean'].mean(), vsl_df['edge_density_mean'].std(),
                    asl_df['edge_density_mean'].mean(), asl_df['edge_density_mean'].std()),
    normalize_metric(vsl_df['motion_mean'].mean(), 'motion_mean',
                    vsl_df['motion_mean'].mean(), vsl_df['motion_mean'].std(),
                    asl_df['motion_mean'].mean(), asl_df['motion_mean'].std()),
    normalize_metric(vsl_df['occlusion_rate'].mean(), 'occlusion_rate',
                    vsl_df['occlusion_rate'].mean(), vsl_df['occlusion_rate'].std(),
                    asl_df['occlusion_rate'].mean(), asl_df['occlusion_rate'].std()),
    normalize_metric(vsl_df['frontality_score'].mean(), 'frontality_score',
                    vsl_df['frontality_score'].mean(), vsl_df['frontality_score'].std(),
                    asl_df['frontality_score'].mean(), asl_df['frontality_score'].std()),
    normalize_metric(vsl_df['person_area_ratio_mean'].mean(), 'person_area_ratio_mean',
                    vsl_df['person_area_ratio_mean'].mean(), vsl_df['person_area_ratio_mean'].std(),
                    asl_df['person_area_ratio_mean'].mean(), asl_df['person_area_ratio_mean'].std()),
]

asl_metrics = [
    normalize_metric(asl_df['blur_mean'].mean(), 'blur_mean',
                    vsl_df['blur_mean'].mean(), vsl_df['blur_mean'].std(),
                    asl_df['blur_mean'].mean(), asl_df['blur_mean'].std()),
    normalize_metric(asl_df['brightness_std'].mean(), 'brightness_std',
                    vsl_df['brightness_std'].mean(), vsl_df['brightness_std'].std(),
                    asl_df['brightness_std'].mean(), asl_df['brightness_std'].std()),
    normalize_metric(asl_df['edge_density_mean'].mean(), 'edge_density_mean',
                    vsl_df['edge_density_mean'].mean(), vsl_df['edge_density_mean'].std(),
                    asl_df['edge_density_mean'].mean(), asl_df['edge_density_mean'].std()),
    normalize_metric(asl_df['motion_mean'].mean(), 'motion_mean',
                    vsl_df['motion_mean'].mean(), vsl_df['motion_mean'].std(),
                    asl_df['motion_mean'].mean(), asl_df['motion_mean'].std()),
    normalize_metric(asl_df['occlusion_rate'].mean(), 'occlusion_rate',
                    vsl_df['occlusion_rate'].mean(), vsl_df['occlusion_rate'].std(),
                    asl_df['occlusion_rate'].mean(), asl_df['occlusion_rate'].std()),
    normalize_metric(asl_df['frontality_score'].mean(), 'frontality_score',
                    vsl_df['frontality_score'].mean(), vsl_df['frontality_score'].std(),
                    asl_df['frontality_score'].mean(), asl_df['frontality_score'].std()),
    normalize_metric(asl_df['person_area_ratio_mean'].mean(), 'person_area_ratio_mean',
                    vsl_df['person_area_ratio_mean'].mean(), vsl_df['person_area_ratio_mean'].std(),
                    asl_df['person_area_ratio_mean'].mean(), asl_df['person_area_ratio_mean'].std()),
]

# Create radar chart
angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
vsl_metrics += vsl_metrics[:1]
asl_metrics += asl_metrics[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

ax.plot(angles, vsl_metrics, 'o-', linewidth=2.5, label='VSL', color='#FF6B6B', markersize=8)
ax.fill(angles, vsl_metrics, alpha=0.25, color='#FF6B6B')

ax.plot(angles, asl_metrics, 'o-', linewidth=2.5, label='ASL', color='#4ECDC4', markersize=8)
ax.fill(angles, asl_metrics, alpha=0.25, color='#4ECDC4')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9)
ax.grid(True, alpha=0.3)

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12, framealpha=0.9)
plt.title('Overall Dataset Quality Assessment\n(Radar Chart - Normalized 0-1)', 
          size=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "quality_radar_chart.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Radar chart saved to: {OUTPUT_DIR / 'quality_radar_chart.png'}")

## Section 9: Final Analysis and Recommendations

In [ ]:
print("\n" + "="*100)
print("FINAL QUALITY ASSESSMENT & RECOMMENDATIONS")
print("="*100)

# Scoring logic: count how many metrics favor each dataset
vsl_wins = 0
asl_wins = 0

scores = []

# 1. Blur/Sharpness (higher is better)
if vsl_df['blur_mean'].mean() > asl_df['blur_mean'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has better sharpness ({vsl_df['blur_mean'].mean():.2f} vs {asl_df['blur_mean'].mean():.2f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has better sharpness ({asl_df['blur_mean'].mean():.2f} vs {vsl_df['blur_mean'].mean():.2f})")

# 2. Lighting Stability (lower is better)
if vsl_df['brightness_std'].mean() < asl_df['brightness_std'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has more stable lighting ({vsl_df['brightness_std'].mean():.2f} vs {asl_df['brightness_std'].mean():.2f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has more stable lighting ({asl_df['brightness_std'].mean():.2f} vs {vsl_df['brightness_std'].mean():.2f})")

# 3. Background Simplicity (lower is better)
if vsl_df['edge_density_mean'].mean() < asl_df['edge_density_mean'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has simpler background ({vsl_df['edge_density_mean'].mean():.4f} vs {asl_df['edge_density_mean'].mean():.4f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has simpler background ({asl_df['edge_density_mean'].mean():.4f} vs {vsl_df['edge_density_mean'].mean():.4f})")

# 4. Camera Stability (lower motion is better)
if vsl_df['motion_mean'].mean() < asl_df['motion_mean'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has more stable camera ({vsl_df['motion_mean'].mean():.4f} vs {asl_df['motion_mean'].mean():.4f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has more stable camera ({asl_df['motion_mean'].mean():.4f} vs {vsl_df['motion_mean'].mean():.4f})")

# 5. Hand Visibility (lower occlusion is better)
if vsl_df['occlusion_rate'].mean() < asl_df['occlusion_rate'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has better hand visibility ({vsl_df['occlusion_rate'].mean():.4f} vs {asl_df['occlusion_rate'].mean():.4f} occlusion)")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has better hand visibility ({asl_df['occlusion_rate'].mean():.4f} vs {vsl_df['occlusion_rate'].mean():.4f} occlusion)")

# 6. Hand Detection Ratio (higher is better)
if vsl_df['hand_visibility_ratio'].mean() > asl_df['hand_visibility_ratio'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has higher hand visibility ratio ({vsl_df['hand_visibility_ratio'].mean():.4f} vs {asl_df['hand_visibility_ratio'].mean():.4f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has higher hand visibility ratio ({asl_df['hand_visibility_ratio'].mean():.4f} vs {vsl_df['hand_visibility_ratio'].mean():.4f})")

# 7. Person Size/Distance (higher is better - person fills frame more)
if vsl_df['person_area_ratio_mean'].mean() > asl_df['person_area_ratio_mean'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has better person size/distance ({vsl_df['person_area_ratio_mean'].mean():.6f} vs {asl_df['person_area_ratio_mean'].mean():.6f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has better person size/distance ({asl_df['person_area_ratio_mean'].mean():.6f} vs {vsl_df['person_area_ratio_mean'].mean():.6f})")

# 8. Frontality/Camera Angle (higher is better - more frontal)
if vsl_df['frontality_score'].mean() > asl_df['frontality_score'].mean():
    vsl_wins += 1
    scores.append(f"✓ VSL has more frontal camera angle ({vsl_df['frontality_score'].mean():.4f} vs {asl_df['frontality_score'].mean():.4f})")
else:
    asl_wins += 1
    scores.append(f"✓ ASL has more frontal camera angle ({asl_df['frontality_score'].mean():.4f} vs {vsl_df['frontality_score'].mean():.4f})")

print("\nMETRIC-BY-METRIC COMPARISON:")
print("-" * 100)
for score in scores:
    print(score)

print("\n" + "-" * 100)
print(f"\nOVERALL SCORE: VSL {vsl_wins} - {asl_wins} ASL (out of 8 metrics)")
print("-" * 100)

# Determine winner
if vsl_wins > asl_wins:
    diff = vsl_wins - asl_wins
    print(f"\n🏆 RECOMMENDATION: VSL is of higher quality (+{diff} point(s))")
    print(f"   VSL is more suitable for sign language recognition models.")
elif asl_wins > vsl_wins:
    diff = asl_wins - vsl_wins
    print(f"\n🏆 RECOMMENDATION: ASL is of higher quality (+{diff} point(s))")
    print(f"   ASL is more suitable for sign language recognition models.")
else:
    print(f"\n🏆 RECOMMENDATION: Both datasets have comparable quality")
    print(f"   Use VSL for efficiency ({len(vsl_df)} videos) or ASL for scale ({len(asl_df)} videos)")

print("\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

# Additional insights
print(f"\n📊 Dataset Scale:")
print(f"   - VSL: {len(vsl_df):,} videos")
print(f"   - ASL: {len(asl_df):,} videos")
print(f"   - ASL has {len(asl_df)/len(vsl_df):.1f}x more videos than VSL")

print(f"\n📐 Resolution:")
print(f"   - VSL: {vsl_df['resolution'].iloc[0] if len(vsl_df) > 0 else 'N/A'}")
print(f"   - ASL: {asl_df['resolution'].iloc[0] if len(asl_df) > 0 else 'N/A'}")

print(f"\n⏱️  Processing Summary:")
print(f"   - Total videos analyzed: {len(vsl_df) + len(asl_df):,}")
print(f"   - Output files: {len(list(OUTPUT_DIR.glob('*.csv'))) + len(list(OUTPUT_DIR.glob('*.json'))) + len(list(OUTPUT_DIR.glob('*.png')))}")

print(f"\n✓ Analysis complete! All results saved to: {OUTPUT_DIR}")
print("="*100)

## Appendix: Pipeline Design Notes

### Design Principles Applied:
1. **No Frame Storage**: All frames processed in-memory using OpenCV's streaming API
2. **Modular Architecture**: Each metric computed by separate function for reusability
3. **MediaPipe Integration**: Hand and Pose detection for accurate metrics
4. **Efficient Batch Processing**: Vectorized NumPy operations for aggregation
5. **Research-Grade Output**: JSON, CSV, and PNG visualizations for reproducibility

### Metrics Explanation:
- **Blur (Laplacian Variance)**: Measures image sharpness; higher = sharper
- **Brightness Std**: Measures lighting consistency; lower = more stable
- **Edge Density**: Background complexity via Canny edges; lower = simpler
- **Optical Flow**: Camera stability; lower = more stable
- **Occlusion Rate**: % frames missing hand detections; lower = better
- **Frontality Score**: Camera angle consistency via pose symmetry; higher = more frontal

### Output Files Generated:
- `dataset_comparison.csv` - Summary table for both datasets
- `vsl_video_metrics.csv` - Detailed metrics for each VSL video
- `asl_video_metrics.csv` - Detailed metrics for each ASL video
- `quality_summary.json` - Aggregated statistics
- `quality_comparison_bars.png` - 6-metric comparison chart
- `quality_comparison_additional.png` - Person area & frontality charts
- `quality_radar_chart.png` - Normalized quality radar chart

### Recommendations for Model Training:
- Use **higher-quality dataset** (more wins in metrics)
- Consider **data augmentation** for lower-quality dataset
- Apply **preprocessing** to normalize lighting/contrast
- Use **hand-tracking loss** to improve occlusion handling